# 01 L1B Structure and Refh

Purpose:
- inspect H5 structure, dimensions, and key datasets;
- explain pulse, sweep, track, `refh`, `rx_waveform`, and `tx_waveform`;
- verify whether `sweep_num` and `track_num` define a complete rectangular acquisition grid.

Scientific scope:
- CASALS L1B is a geolocated waveform product;
- each pulse has one official geolocated `refh` point;
- `refh` is associated with the maximum-amplitude RX waveform bin;
- this notebook does not perform waveform decomposition or create a multi-return point cloud.

In [ ]:
from pathlib import Path
import sys

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_casals_l1b_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "waveform_components.py").exists():
            return candidate
        if (candidate / "CASALS_L1B" / "waveform_components.py").exists():
            return candidate / "CASALS_L1B"
    raise FileNotFoundError("Could not locate CASALS_L1B directory from the current notebook working directory.")


CASALS_L1B_DIR = find_casals_l1b_root()
if str(CASALS_L1B_DIR) not in sys.path:
    sys.path.insert(0, str(CASALS_L1B_DIR))

from waveform_components import build_record_index_grid, read_attrs_subset, require_dataset

H5_PATH = CASALS_L1B_DIR / "casals_h5_downloads" / "casals_l1b_20241118T171757_001_02.h5"
assert H5_PATH.exists(), H5_PATH

In [ ]:
with h5py.File(H5_PATH, "r") as h5:
    index_info = build_record_index_grid(h5)
    attrs_subset = read_attrs_subset(h5)
    refh_lon = np.asarray(require_dataset(h5, "refh_longitude")[...], dtype=np.float64)
    refh_lat = np.asarray(require_dataset(h5, "refh_latitude")[...], dtype=np.float64)
    refh = np.asarray(require_dataset(h5, "refh")[...], dtype=np.float64)
    refh_amp = np.asarray(require_dataset(h5, "refh_amp")[...], dtype=np.float64)
    refh_snr = np.asarray(require_dataset(h5, "refh_snr")[...], dtype=np.float64)
    good_snr = np.asarray(require_dataset(h5, "good_snr")[...], dtype=bool)
    rx_shape = tuple(require_dataset(h5, "rx_waveform").shape)
    tx_shape = tuple(require_dataset(h5, "tx_waveform").shape)
    rx_bins = np.asarray(require_dataset(h5, "rx_bins")[...], dtype=np.int64)
    tx_bins = np.asarray(require_dataset(h5, "tx_bins")[...], dtype=np.int64)

summary = pd.Series(
    {
        "h5_path": str(H5_PATH.resolve()),
        "n_records": index_info.n_records,
        "n_sweeps": index_info.n_sweeps,
        "n_tracks": index_info.n_tracks,
        "complete_rectangular_grid": index_info.complete_rectangular_grid,
        "duplicate_sweep_track_cells": index_info.duplicate_sweep_track_cells,
        "missing_sweep_track_cells": index_info.missing_sweep_track_cells,
        "rx_waveform_shape": rx_shape,
        "tx_waveform_shape": tx_shape,
        "n_rx_bins": rx_bins.size,
        "n_tx_bins": tx_bins.size,
    }
)
display(summary.to_frame("value"))
display(pd.Series(attrs_subset, name="attr_value").to_frame())

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10), constrained_layout=True)
sample_idx = np.random.default_rng(42).choice(refh_lon.size, size=min(200_000, refh_lon.size), replace=False)

axes[0, 0].scatter(refh_lon[sample_idx], refh_lat[sample_idx], c=refh_snr[sample_idx], s=0.2, cmap="viridis", linewidths=0)
axes[0, 0].set_title("Refh footprint colored by refh_snr")
axes[0, 0].set_xlabel("Longitude")
axes[0, 0].set_ylabel("Latitude")

axes[0, 1].hist(refh_snr[np.isfinite(refh_snr)], bins=100, color="tab:green")
axes[0, 1].set_title("refh_snr distribution")
axes[0, 1].set_xlabel("refh_snr")

axes[0, 2].hist(refh_amp[np.isfinite(refh_amp)], bins=100, color="tab:blue")
axes[0, 2].set_title("refh_amp distribution")
axes[0, 2].set_xlabel("refh_amp")

axes[1, 0].hist(refh[np.isfinite(refh)], bins=100, color="tab:red")
axes[1, 0].set_title("refh distribution")
axes[1, 0].set_xlabel("refh")

axes[1, 1].bar(["good_snr=False", "good_snr=True"], [np.count_nonzero(~good_snr), np.count_nonzero(good_snr)], color=["tab:gray", "tab:purple"])
axes[1, 1].set_title("good_snr counts")

axes[1, 2].plot(tx_bins, np.arange(tx_bins.size), color="tab:orange")
axes[1, 2].set_title("TX/RX bin coordinate arrays")
axes[1, 2].plot(rx_bins, np.linspace(0, tx_bins.size - 1, rx_bins.size), color="tab:blue", alpha=0.7)
axes[1, 2].set_xlabel("Bin value")
axes[1, 2].set_ylabel("Reference axis")

plt.show()

This notebook is the entry point for the waveform diagnostics workflow. It explains the storage structure and the single-`refh` scientific model, but it does not attempt waveform decomposition or secondary-return georeferencing.